# 06 · After reversibility, the head is the wall

Notebook 04's exact peaks landed in the loss (`_log_softmax_backward_data`) or in one layer's attention
softmax, never in the stack's stored activations, because the reversible stack no longer stores any. What
is left grows with the batch through the output head: one fp32 logits tensor is 512 x 8,192 x 4 bytes =
16 MiB per sequence, and the loss keeps several of them alive at once (logits, log-softmax, and its gradient).

This notebook changes one thing: the head and loss are computed 8 sequences at a time and recomputed in
the backward pass (`--loss-chunk 8`, gradient identical to the plain head to 2e-16 in float64). It then
repeats notebook 04's ceiling search for both stacks. No 50M-token training here, only the 4-step probes.

In [1]:
import sys, os, json, pathlib, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not pathlib.Path("ERA_V5").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/amukka/ERA_V5.git"], check=True)
    os.chdir("ERA_V5/session13_Distributed_Training")
ROOT = pathlib.Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
if not (ROOT / "data" / "train.bin").exists():          # ~5 minutes: stream FineWeb-Edu, train the BPE
    subprocess.run([sys.executable, "tools/prepare_data.py"], check=True)

import torch, platform
from src import experiments as X
from src.train import pick_device
dev = pick_device()
print("device:", dev, "| torch", torch.__version__, "| python", platform.python_version(), "|", platform.platform())
if dev == "mps":
    print(f"MPS recommended working set: {torch.mps.recommended_max_memory()/2**30:.2f} GiB")
if dev == "cuda":
    print(torch.cuda.get_device_name(), f"{torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB")
print(json.dumps(json.loads((ROOT / "data" / "meta.json").read_text()), indent=1))

device: mps | torch 2.13.0 | python 3.14.5 | macOS-26.6.2-arm64-arm-64bit-Mach-O
MPS recommended working set: 11.84 GiB
{
 "source": "HuggingFaceFW/fineweb-edu, config sample-10BT, streamed in order",
 "vocab_size": 8192,
 "eos_id": 0,
 "train_tokens": 56000000,
 "train_docs": 46080,
 "val_tokens": 1000000,
 "val_docs": 1024,
 "chars_per_token": 3.886,
 "note": "val.bin comes from the tokenizer's training documents; train.bin from the documents after them",
 "seconds": 314.3
}


In [2]:
chosen = json.loads((X.RESULTS / "chosen_variant.json").read_text())
ex = ("--loss-chunk", "8")
print("standard, chunked head")
m_std, t_std = X.max_batch("standard", 1.0, start=32, step=4, extra=ex)
print(chosen["variant"] + ", chunked head")
m_rev, t_rev = X.max_batch(chosen["variant"], chosen["h"], start=64, step=16, extra=ex)
old = json.loads((X.RESULTS / "max_batch.json").read_text())
print(f"\nmax batch with a chunked head: standard {m_std} (was {old['standard']['max_batch']}), "
      f"{chosen['variant']} {m_rev} (was {old[chosen['variant']]['max_batch']}); ratio {m_rev/m_std:.2f}x")
(X.RESULTS / "max_batch_chunked_head.json").write_text(json.dumps(
    {"standard": {"max_batch": m_std, "trail": t_std},
     chosen["variant"]: {"max_batch": m_rev, "trail": t_rev, "h": chosen["h"]}}, indent=2))

standard, chunked head


  standard  batch   32  peak    7,898 MiB   10,270 tok/s


  standard  batch   64  does not fit: RuntimeError: MPS backend out of memory (MPS allocated: 12.02 GiB, other a


  standard  batch   48  peak    9,460 MiB   10,226 tok/s


  standard  batch   56  does not fit: RuntimeError: MPS backend out of memory (MPS allocated: 12.03 GiB, other a


...

  momentum  batch  320  does not fit: RuntimeError: MPS backend out of memory (MPS allocated: 11.59 GiB, other a


  momentum  batch  288  does not fit: RuntimeError: MPS backend out of memory (MPS allocated: 10.94 GiB, other a


  momentum  batch  272  does not fit: RuntimeError: MPS backend out of memory (MPS allocated: 9.02 GiB, other al



max batch with a chunked head: standard 48 (was 32), momentum 256 (was 88); ratio 5.33x


4143